# **1. Introduction & Environmental Setup**
>This analysis investigates Land Surface Temperature(LST) patterns for Dar es Salaam and Singapore using MODIS satellite data. We focus on comparing the Urban Heat Island(UHI) Intensity and thermal homogeneity across these two distinct tropical enviroments.

In [1]:
import ee
import geemap
import datetime

In [2]:
ee.Authenticate()

True

In [3]:
ee.Initialize(project='black-octagon-291810')

Area of Interest

In [4]:
# Dar es Salaam
aoi_dar = ee.FeatureCollection("projects/sat-io/open-datasets/FAO/GAUL/GAUL_2024_L1") \
            .filter(ee.Filter.eq("gaul1_name", "Dar Es Salaam"))

# Singapore
aoi_sgp = ee.FeatureCollection("projects/sat-io/open-datasets/FAO/GAUL/GAUL_2024_L0") \
            .filter(ee.Filter.eq("gaul0_name", "Singapore"))

dar_geom = aoi_dar.geometry()
sgp_geom = aoi_sgp.geometry()


# **2. Data Processing Pipeline**

> To ensure accuracy, we convert MODIS LST values from Kelvin to Celsius and apply a scaling factor of 0.02. We then generate annual median composites to fill data gaps caused by cloud cover

In [5]:
# Scaling and Conversion Function
def prepare_lst(image):
    # Scale factor for MODIS LST is 0.02, then convert Kelvin to Celsius
    # Band 'LST_Day_1km' is used for daytime surface temperature
    processed = image.multiply(0.02).subtract(273.15)
    return processed.copyProperties(image, ['system:time_start'])

# Function to get Annual Median LST
def get_annual_lst(geom, year):
    start_date = f'{year}-01-01'
    end_date = f'{year}-12-31'

    collection = ee.ImageCollection("MODIS/061/MOD11A2") \
        .filterBounds(geom) \
        .filterDate(start_date, end_date) \
        .select('LST_Day_1km')

    # Calculate median to fill gaps and clip AOI
    return collection.map(prepare_lst).median().clip(geom)

# Generate LST for Dar es Salaam (2020 vs 2025)
lst_dar_2020 = get_annual_lst(dar_geom, 2020)
lst_dar_2025 = get_annual_lst(dar_geom, 2025)
dar_increase = lst_dar_2025.subtract(lst_dar_2020)

# Generate LST for Singapore (2020 vs 2025)
lst_sgp_2020 = get_annual_lst(sgp_geom, 2020)
lst_sgp_2025 = get_annual_lst(sgp_geom, 2025)
sgp_increase = lst_sgp_2025.subtract(lst_sgp_2020)


# **Spatial Visualization (The Map)**

> By mapping the 2025 LST data, we can visually identify 'hot spots.' In Dar es Salaam, these typically correlate with high-density built environments, while cooler areas represent coastal zones or inland vegetation

In [6]:
# Calculate the actual change in Celsius
dar_change = lst_dar_2025.subtract(lst_dar_2020)
sgp_change = lst_sgp_2025.subtract(lst_sgp_2020)

In [7]:
# Calculate UHI Intensity (Pixel Temp - City Mean Temp)
dar_mean = lst_dar_2025.reduceRegion(ee.Reducer.mean(), dar_geom, 1000).get('LST_Day_1km')
dar_uhi = lst_dar_2025.subtract(ee.Image.constant(dar_mean))

In [8]:
# Calculate the City-Wide Mean for Singapore 2025
sgp_mean = lst_sgp_2025.reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=sgp_geom,
    scale=1000,
    maxPixels=1e9
).get('LST_Day_1km')

# Subtract the City Mean from every pixel to get Intensity
sgp_uhi = lst_sgp_2025.subtract(ee.Image.constant(sgp_mean))

In [9]:
# Visualization
Map = geemap.Map()

# LST Range
lst_vis = {
    'min': 22,
    'max': 40,
    'palette': ['blue', 'green', 'yellow', 'orange', 'red']
}

# Change Range (To highlight the increase/decrease in °C)
diff_vis = {
    'min': -2,
    'max': 5,
    'palette': ['blue', 'white', 'red']
}

# Temperature Change Pallete
change_vis = {
    'min': -3,
    'max': 3,
    'palette': ['#0000FF', '#FFFFFF', '#FF0000'] # Blue (Cooling), White (No Change), Red (Heating)
}

# Dar es Salaam Layers
Map.addLayer(lst_dar_2020, lst_vis, 'Dar es Salaam LST 2020 (Celsius)')
Map.addLayer(lst_dar_2025, lst_vis, 'Dar es Salaam LST 2025 (Celsius)')

# Singapore Layers
Map.addLayer(lst_sgp_2020, lst_vis, 'Singapore LST 2020 (Celsius)')
Map.addLayer(lst_sgp_2025, lst_vis, 'Singapore LST 2025 (Celsius)')

# Temperature Increase Layers (The difference)
Map.addLayer(dar_increase, diff_vis, 'Dar Temp Increase (2020-2025)')
Map.addLayer(sgp_increase, diff_vis, 'Singapore Temp Increase (2020-2025)')

# Temperature Change
Map.addLayer(dar_change, change_vis, 'Dar es Salaam Temp Change')
Map.addLayer(sgp_change, change_vis, 'Singapore Temp Change')

# UHI Intensity
Map.addLayer(dar_uhi, {'min': -5, 'max': 5, 'palette': ['blue', 'white', 'red']}, 'Dar UHI Intensity')
Map.addLayer(sgp_uhi, {'min': -5, 'max': 5, 'palette': ['blue', 'white', 'red']}, 'Singapore UHI Intensity')

# Set center
Map.centerObject(dar_geom, 10)
Map

Map(center=[-6.885315881886399, 39.261678578380916], controls=(WidgetControl(options=['position', 'transparent…

# **Comparative Statistics**

***Standard Deviation***

**This measures how much the temperature varies across the city.**

*   High Std Dev (Singapore): Indicates a "diverse" thermal landscape. It means the city has very hot areas but also very effective "green lung" cooling zones that create a big temperature spread.
*   Low Std Dev (Dar es Salaam): Indicates "thermal homogeneity". The heat is more "smeared" or consistent across the city, suggesting a lack of large, distributed cooling parks.

***Max UHI Intensity***

**It identifies the single hottest urban pixel compared to the city’s average**

Even though Singapore is more developed, Dar es Salaam has higher peak intensity. This suggests that Dar's "hot spots" (likely dense unplanned settlements with metal roofs) are trapping more extreme heat relative to their surroundings than Singapore's skyscrapers are.

In [10]:
def get_advanced_stats(uhi_image, geom, city_name):
    # Calculate Max UHI Intensity
    max_val = uhi_image.reduceRegion(
        reducer=ee.Reducer.max(),
        geometry=geom,
        scale=1000,
        maxPixels=1e9
    ).get('LST_Day_1km')

    # Calculate Standard Deviation (Thermal Homogeneity)
    std_dev = uhi_image.reduceRegion(
        reducer=ee.Reducer.stdDev(),
        geometry=geom,
        scale=1000,
        maxPixels=1e9
    ).get('LST_Day_1km')

    return {
        'City': city_name,
        'Max_UHI_Intensity': max_val.getInfo(),
        'Std_Dev': std_dev.getInfo()
    }


dar_stats = get_advanced_stats(dar_uhi, dar_geom, "Dar es Salaam")
sgp_stats = get_advanced_stats(sgp_uhi, sgp_geom, "Singapore")


print(f"{dar_stats['City']} | Max UHI Intensity: {dar_stats['Max_UHI_Intensity']:.2f}°C | Std Dev: {dar_stats['Std_Dev']:.2f}")
print(f"{sgp_stats['City']} | Max UHI Intensity: {sgp_stats['Max_UHI_Intensity']:.2f}°C | Std Dev: {sgp_stats['Std_Dev']:.2f}")

Dar es Salaam | Max UHI Intensity: 4.57°C | Std Dev: 1.32
Singapore | Max UHI Intensity: 4.25°C | Std Dev: 1.70


***Thermal Spread***

**This is the distance between the absolute coldest pixel and the absolute hottest pixel in the study area.**

This confirms that Dar es Salaam experiences a wider range of temperature extremes across its entire geography than Singapore does.



In [11]:
def get_range(image, geom, name):
    stats = image.reduceRegion(
        reducer=ee.Reducer.minMax(),
        geometry=geom,
        scale=1000
    )
    # Get the Max and Min to find the spread
    mx = stats.get('LST_Day_1km_max').getInfo()
    mn = stats.get('LST_Day_1km_min').getInfo()
    print(f"{name} Thermal Spread: {round(mx - mn, 2)}°C")

get_range(lst_dar_2025, dar_geom, "Dar es Salaam")
get_range(lst_sgp_2025, sgp_geom, "Singapore")

Dar es Salaam Thermal Spread: 8.28°C
Singapore Thermal Spread: 7.96°C
